In [ ]:
!pip install scikit-learn gensim nltk bertopic sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.4 MB/s eta 0:00:00


In [ ]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(
    subset='all',
    remove=('headers','footers','quotes')
)

documents = data.data
print(len(documents))



18846


In [ ]:
import nltk
import re
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)    #
    tokens = [w for w in text.split() if w not in stop_words]

    # for w in text.split()
    #if w not in stop_words
    #print(w)

    return " ".join(tokens)

documents_clean = [preprocess(doc) for doc in documents]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(documents_clean)

lsa_model = TruncatedSVD(n_components=10, random_state=42)
lsa_topics = lsa_model.fit_transform(X)


In [ ]:
terms = vectorizer.get_feature_names_out()

for i, comp in enumerate(lsa_model.components_):
    terms_comp = zip(terms, comp)
    sorted_terms = sorted(terms_comp, key=lambda x: x[1], reverse=True)[:10]
    print(f"Topic {i}: ", [t[0] for t in sorted_terms])


Topic 0:  ['would', 'one', 'dont', 'like', 'know', 'get', 'people', 'think', 'im', 'could']
Topic 1:  ['thanks', 'windows', 'please', 'email', 'anyone', 'drive', 'card', 'file', 'dos', 'advance']
Topic 2:  ['god', 'thanks', 'please', 'email', 'anyone', 'know', 'jesus', 'advance', 'would', 'bible']
Topic 3:  ['thanks', 'game', 'please', 'email', 'anyone', 'games', 'team', 'im', 'know', 'year']
Topic 4:  ['drive', 'god', 'scsi', 'mb', 'please', 'card', 'jesus', 'drives', 'sale', 'ide']
Topic 5:  ['windows', 'god', 'game', 'file', 'dos', 'games', 'team', 'files', 'im', 'window']
Topic 6:  ['would', 'know', 'im', 'dont', 'like', 'anyone', 'think', 'drive', 'get', 'ive']
Topic 7:  ['would', 'game', 'team', 'games', 'could', 'key', 'chip', 'drive', 'players', 'appreciated']
Topic 8:  ['would', 'god', 'car', 'new', 'like', 'bike', 'good', 'sale', 'jesus', 'price']
Topic 9:  ['would', 'drive', 'windows', 'car', 'armenian', 'new', 'armenians', 'dos', 'people', 'israel']


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

count_vectorizer = CountVectorizer(max_features=5000)
X_counts = count_vectorizer.fit_transform(documents_clean)


In [ ]:
lda_model = LatentDirichletAllocation(
    n_components=10,
    random_state=42
)
lda_model.fit(X_counts)


LatentDirichletAllocation(random_state=42)

In [ ]:
words = count_vectorizer.get_feature_names_out()

for idx, topic in enumerate(lda_model.components_):
    print(f"Topic {idx}: ",
          [words[i] for i in topic.argsort()[-10:]])


Topic 0:  ['using', 'thanks', 'drive', 'would', 'one', 'system', 'get', 'windows', 'file', 'use']
Topic 1:  ['email', 'image', 'send', 'new', 'also', 'information', 'data', 'list', 'available', 'space']
Topic 2:  ['security', 'chip', 'one', 'jews', 'encryption', 'use', 'israel', 'would', 'government', 'key']
Topic 3:  ['play', 'think', 'good', 'last', 'one', 'would', 'games', 'year', 'team', 'game']
Topic 4:  ['think', 'know', 'say', 'believe', 'jesus', 'people', 'would', 'one', 'god', 'maxaxaxaxaxaxaxaxaxaxaxaxaxaxax']
Topic 5:  ['control', 'disease', 'one', 'also', 'study', 'states', 'medical', 'health', 'use', 'gun']
Topic 6:  ['san', 'period', 'ms', 'university', 'said', 'la', 'stephanopoulos', 'new', 'president', 'mr']
Topic 7:  ['government', 'know', 'right', 'us', 'like', 'one', 'think', 'dont', 'would', 'people']
Topic 8:  ['think', 'time', 'good', 'im', 'know', 'dont', 'get', 'like', 'would', 'one']
Topic 9:  ['images', 'space', 'gif', 'first', 'one', 'turkish', 'image', 'jpeg

In [ ]:
import gensim
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

texts = [doc.split() for doc in documents_clean]
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

lda_gensim = gensim.models.ldamodel.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,
    random_state=42
)

coherence_model = CoherenceModel(
    model=lda_gensim,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

print("Coherence Score:", coherence_model.get_coherence())

Coherence Score: 0.5372364443938478


In [ ]:
from sklearn.decomposition import NMF
# Performing NMF
nmf = NMF(n_components=3, random_state=42)
nmf.fit(X)
# Displaying NMF topics
terms = vectorizer.get_feature_names_out()

print("\nNMF Topics:\n")
for i, comp in enumerate(nmf.components_):
    words = [terms[j] for j in comp.argsort()[-10:]]
    print(f"Topic {i+1}: {words}")


NMF Topics:

Topic 1: ['time', 'good', 'well', 'get', 'people', 'like', 'think', 'one', 'dont', 'would']
Topic 2: ['file', 'use', 'card', 'know', 'drive', 'email', 'anyone', 'windows', 'please', 'thanks']
Topic 3: ['us', 'christians', 'faith', 'christ', 'people', 'christian', 'bible', 'believe', 'jesus', 'god']
